# Lab 23: FedSpeak Analysis — Guided Lab
## ECON 3916: Applied Machine Learning for Economists
### Guided Construction Lab | 60 min Core + 22 min Write it yourself + 12 min Extension + take-home Challenge

---

**Learning Objectives:**
- Tokenize and preprocess FOMC meeting minutes for NLP analysis
- Build a TF-IDF document-term matrix from real central bank text, and compute one TF-IDF score by hand
- Compute sentiment scores with short word lists in the style of Loughran-McDonald (LM)
- Visualize sentiment trends across time and check them against policy events
- Write a function that gives the TF-IDF score of any word in any document
- (Extension) Cluster FOMC documents with K-Means on TF-IDF vectors (Ch 22 callback)
- (Challenge) Compare pre-2020 vs post-2020 sentiment patterns

**Dataset:** FOMC meeting minutes: 243 meetings, February 2000 to July 2026, from the HuggingFace
dataset `vtasca/fomc-statements-minutes`, saved on 4 October 2026 so everyone works on the same text.

**Prerequisites:** Chapter 6 (feature engineering), and Lab 22 (K-Means and the silhouette), which
the optional Part 6 uses again.

**How this lab works:** Parts 1, 2, 3 and 5 are GUIDED: run each cell as it is and read the output.
Part 4 has blanks (`____`) for you to fill. "Write it yourself" is required: a short function
you write from a blank cell. Part 6 is an optional extension, and the Challenge is a take-home.

**Plan on about 1¾ hours** (1 hour 40 to 1 hour 45 minutes) for the required work: setup, Parts 1-5,
Write it yourself, and the README and submission. Cleaning the 243 documents in Part 2 takes 10-30
seconds; every other cell runs in a few seconds.

**AI in this lab:** you may ask Colab to *explain* an error or a line of code (click **Explain error**, or select a line and ask Gemini to explain it). Do not ask it to write or fix code, and turn off AI code completion. This lab has no AI coding section: you write every line yourself. The one use of AI is drafting your README from the prompt at the end.

---


> ### What you write, and what you run
>
> **You write:**
> - **Eight blanks (`____`)** in one cell, Step 4: which word list each count uses, and the two sentiment formulas. Each is a name.
> - **Write it yourself (required),** after Part 5: `tfidf_score()`, a function that gives back the TF-IDF score of one word in one document. About five lines from a blank cell. The check cell below it must print "Passed"; it then prints your function's scores for four words in three meetings.
> - **One written answer,** in a text cell after Write it yourself.
>
> **You run and read:** everything else. Setup and Parts 1-5 are working code to learn from. Read each cell before you run it, and make sure you can say what each line does. The CHECK cell after Write it yourself is different: you only run it and read its message; you do not need to follow its code.
>
> **Part 6** (optional) has two blanks and one question of its own. **The Challenge** (take-home) asks for a chart and a short paragraph. The **README** is text only: the prompt at the end is the one place you use AI, and only for documentation.


## Setup

Colab comes with scikit-learn and NLTK (the Natural Language Toolkit). The first line makes sure of
both and installs `datasets`, HuggingFace's package, which only the optional live download in Part 1
uses. The `!` runs that line as a terminal command, not as Python.

NLTK keeps its word lists and dictionaries as separate data files, and `nltk.download()` fetches the
ones this lab uses: the tokenizer's rules, the English stop-word list, and the WordNet dictionary that
the lemmatizer looks words up in. `quiet=True` hides the download messages.


In [ ]:
# GUIDED — run as-is
# Setup: install the packages, import them, and download NLTK's data files
!pip install datasets nltk scikit-learn -q

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import nltk
nltk.download('punkt_tab', quiet=True)     # the tokenizer's rules
nltk.download('stopwords', quiet=True)     # lists of common words such as "the" and "and"
nltk.download('wordnet', quiet=True)       # the dictionary behind the lemmatizer
nltk.download('omw-1.4', quiet=True)       # extra WordNet data that some NLTK versions also ask for

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD, PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

print('Libraries loaded. Ready to analyze FedSpeak.')

## Part 1: Load FOMC Minutes (GUIDED — 6 min)

The Federal Open Market Committee (FOMC) sets U.S. interest rates. It meets eight times a year, more
often in a crisis, and a few weeks after each meeting it publishes **minutes**: a long account of what
the staff reported and what the members said. Each document here is one meeting's minutes.

**Why FOMC minutes?** Markets read the Fed's wording closely. In March 2015, dropping the word
"patient" from the FOMC's statement was read as opening the door to the first rate rise since 2006.
NLP lets us measure such shifts across hundreds of documents instead of reading each one.

The minutes come from the HuggingFace dataset `vtasca/fomc-statements-minutes`, which also holds the
FOMC's shorter policy statements. Its minutes were saved on 4 October 2026 as one compressed CSV file,
with one row per meeting, in date order: the `date` and the `text` of the minutes. The live dataset
gains a row whenever new minutes come out, so the saved copy keeps your numbers equal to the expected
outputs below.

- `pd.read_csv` reads the `.gz` file directly: it sees the ending and unzips the file as it reads.
- `try` runs the lines under it; if they raise an error (no internet, say), `except` runs instead
  and reads a copy of the file from a `data` folder beside the notebook. That copy exists only in a
  downloaded copy of the course repository, not in Colab: in Colab, Step 1 needs the internet.
- `pd.to_datetime` turns the date strings into dates, and `.dt.year` takes the year of each.
- `.str.split()` splits each document at spaces and line breaks, and `.str.len()` counts the pieces:
  a rough word count.
- `', '.join(...)` glues the first five dates into one string.


In [ ]:
# GUIDED — run as-is
# Step 1: load the saved FOMC minutes, one row per meeting
DATA_URL = ('https://raw.githubusercontent.com/RyanPiao/econ3916-student-notebooks/main/'
            '3916/data/lab-ch23-fomc-minutes-2026-10.csv.gz')
LOCAL_COPY = 'data/lab-ch23-fomc-minutes-2026-10.csv.gz'

try:
    fomc_df = pd.read_csv(DATA_URL)
except Exception:                       # works only in a downloaded copy of the course repository
    fomc_df = pd.read_csv(LOCAL_COPY)

fomc_df['date'] = pd.to_datetime(fomc_df['date'])
fomc_df['year'] = fomc_df['date'].dt.year
fomc_df['word_count'] = fomc_df['text'].str.split().str.len()

print(f'Total documents: {len(fomc_df)}')
print(f'Date range: {fomc_df["date"].min().date()} to {fomc_df["date"].max().date()}')
print(f'Average word count: {fomc_df["word_count"].mean():.0f}')
print(f'Min / Max word count: {fomc_df["word_count"].min()} / {fomc_df["word_count"].max()}')
print('Sample meeting dates:', ', '.join(str(d.date()) for d in fomc_df['date'].head(5)))

**Expected output:** `Total documents: 243`, dates from 2000-02-02 to 2026-07-29, an average of 7118
words (from 3237 to 14364), and the first five meeting dates, starting 2000-02-02, 2000-03-21.

> **Live data (optional, not for the submitted notebook).** To work on the dataset as it is today, in
> a copy of this notebook, replace Step 1's `try`/`except` with:
> ```python
> from datasets import load_dataset
> fomc_df = pd.DataFrame(load_dataset('vtasca/fomc-statements-minutes', split='train'))
> fomc_df = fomc_df[fomc_df['Type'] == 'Minute'].rename(columns={'Date': 'date', 'Text': 'text'})
> fomc_df = fomc_df.sort_values('date').reset_index(drop=True)
> ```
> Every number from here on may then differ from the expected outputs.


## Part 2: Preprocess Text (GUIDED — 10 min)

Before computing TF-IDF, we clean and normalize the text:
1. **Lowercase**: "Federal" and "federal" become the same token
2. **Remove everything but letters and spaces**: numbers and punctuation go
3. **Tokenize**: split the text into words, called tokens
4. **Remove stop words and short words**: drop common words such as "the", "is" and "and", and every
   word of one or two letters
5. **Lemmatize**: reduce a word to its dictionary form: "rates" becomes "rate", "economies" becomes
   "economy". Given no part of speech, WordNet's lemmatizer treats every word as a noun, so mostly
   plurals change: "running" stays "running"

Each decision is a **feature engineering choice** (Chapter 6) that affects downstream results.

- `re.sub(r'[^a-z\s]', '', text)` replaces every character that is not (`^`) a letter `a-z` or white
  space `\s` with nothing.
- `[t for t in tokens if ...]` is a list comprehension: it builds a new list of the tokens that pass
  the test after `if`.
- `stop_words` is a set: Python checks `t not in stop_words` much faster on a set than on a list.
- `' '.join(tokens)` glues the tokens back into one string with a space between each, so each
  document stays one string.


In [ ]:
# GUIDED — run as-is
# Step 2: a function that cleans one document
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    """Lowercase, keep letters only, tokenize, drop stop words and short words, lemmatize."""
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in stop_words and len(t) > 2]
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return ' '.join(tokens)

print(f'Stop words in the list: {len(stop_words)}')

**Expected output:** `Stop words in the list: 198`. NLTK has changed this list over the years: newer
lists add words with apostrophes ("doesn't"), which Part 2 removes, so the counts do not change.

Step 2, continued runs the function on every document: `.apply(preprocess_text)` calls it on each
entry of the column in turn, which takes 10-30 seconds. `[:200]` keeps the first 200 characters.


In [ ]:
# GUIDED — run as-is
# Step 2, continued: clean every document, and compare the first one before and after
fomc_df['clean_text'] = fomc_df['text'].apply(preprocess_text)

print('=== Before Preprocessing (first 200 chars) ===')
print(fomc_df['text'].iloc[0][:200])
print('\n=== After Preprocessing (first 200 chars) ===')
print(fomc_df['clean_text'].iloc[0][:200])
print(f'\nTokens in first document: {len(fomc_df["clean_text"].iloc[0].split())}')

**Expected output:** the raw text opens with a title and many blank lines; the cleaned text starts
`minute federal open market committee february meeting ...`, and `Tokens in first document: 4832`.

Look for `presentmr` near the end of the cleaned text. The original reads `Present:Mr. Greenspan`,
with no space after the colon, so removing the punctuation glued two words into one. Every hyphenated
word goes the same way: "mortgage-backed" becomes `mortgagebacked`.


## Part 3: Build TF-IDF Matrix (GUIDED — 15 min)

TF-IDF turns each document into a row of numbers, one per term. A term's score in a document is high
when the term is frequent in that document (TF, term frequency) and rare across the documents (IDF,
inverse document frequency). The textbook version, the one in the lecture:

$$\text{TF-IDF}(w, d) = \underbrace{\frac{\text{count of } w \text{ in } d}{\text{tokens in } d}}_{\text{TF}} \times \underbrace{\ln\frac{N}{\text{number of documents that use } w}}_{\text{IDF}}$$

where $N$ is the number of documents. scikit-learn's `TfidfVectorizer` uses a variant, so its numbers
differ from this formula:
- its TF is the raw count, not the share; it then scales each document's row to length 1, which
  divides out the document's length, so count and share end up equivalent;
- its IDF is $\ln\frac{N+1}{\text{df}+1} + 1$, where df is the number of documents that use the word:
  1 added inside the log and 1 added to the log.

Step 3, continued computes the textbook formula by hand.

**The settings.** Step 3 sets three limits on the vocabulary, so that it keeps:
- `min_df`: only terms used in at least **5 meetings' minutes**, which drops typos and one-off names;
- `max_df`: no term used in more than **85% of the minutes**: a term used nearly everywhere is
  background, not signal;
- `max_features`: at most the **5,000 terms used most often** across all the minutes.

`min_df` and `max_df` each take either a whole number (a count of documents) or a decimal between 0
and 1 (a share of documents): `min_df=5` is a count, five meetings, and `max_df=0.85` is a share.

`ngram_range=(1, 2)` keeps single words and pairs of neighbouring words
("federal fund"). `fit_transform` learns the vocabulary and builds the matrix in one go. The matrix is
**sparse**: it stores only the entries that are not zero, and `.nnz` counts them, so
`1 - nnz / (rows × columns)` is the share of zeros. `get_feature_names_out()` gives the terms, in
alphabetical order.


In [ ]:
# GUIDED — run as-is
# Step 3: build the TF-IDF matrix from the cleaned minutes
tfidf = TfidfVectorizer(
    min_df=5,                 # a count: used in at least 5 documents
    max_df=0.85,              # a share: used in at most 85% of them
    max_features=5000,        # the 5,000 terms used most often
    ngram_range=(1, 2),       # single words and two-word pairs
)
tfidf_matrix = tfidf.fit_transform(fomc_df['clean_text'])
feature_names = tfidf.get_feature_names_out()

print(f'TF-IDF matrix shape: {tfidf_matrix.shape}')
print(f'  → {tfidf_matrix.shape[0]} documents × {tfidf_matrix.shape[1]} terms')
print(f'Sparsity: {1 - tfidf_matrix.nnz / (tfidf_matrix.shape[0] * tfidf_matrix.shape[1]):.2%}')
print(f'\nSample vocabulary terms: {list(feature_names[:10])}')

**Expected output:** `TF-IDF matrix shape: (243, 5000)`, `Sparsity: 69.14%`, and ten terms from
`abate` to `accept`. If you try other settings, these are the slips to know: `69.11%` means `min_df`
is a share (0.05); 5 meetings is a count. `82.38%` means `max_df` counts 85 documents rather than 85%
of them, and `(243, 36164)` with `91.88%` means `max_features` is missing.

69% is far less sparse than the 95-99% usual for short texts such as tweets: these documents are long,
the same committee writes the same kind of document every time, and `max_features` kept only the
commonest terms (without it, 91.88% of the entries are zeros).

Step 3, continued asks which terms carry the most weight on average. `tfidf_matrix.mean(axis=0)`
averages each column (each term) over the 243 documents; `np.asarray(...).ravel()` turns the result
into a flat array; `pd.Series(..., index=feature_names)` labels each average with its term, and
`.nlargest(10)` keeps the ten largest. `.items()` gives each label with its value, and `:25s` pads a
term to 25 characters so the numbers line up.


In [ ]:
# GUIDED — run as-is
# Step 3, continued: the 10 terms with the highest average TF-IDF
mean_tfidf = pd.Series(np.asarray(tfidf_matrix.mean(axis=0)).ravel(), index=feature_names)

print('Top 10 terms by average TF-IDF across all documents:')
for term, score in mean_tfidf.nlargest(10).items():
    print(f'  {term:25s} {score:.4f}')

**Expected output:** `messrs` (0.0547) first, then `july`, `loan`, `january`, ..., down to `april`
(0.0437). Eight of the ten are month names, and `messrs`, the plural of "Mr.", comes from the list of
who attended. Part 2 removed the digits from dates but kept the month names, and no month name except
"may" is used in more than 85% of the minutes, so `max_df` kept them. The strongest signal in the
matrix is partly *when* a meeting happened, not what was said.

Now the formula by hand, for two words in the minutes of 16 December 2008, the meeting that cut the
federal funds rate to near zero.

- `fomc_df.loc[fomc_df['date'] == '2008-12-16', 'clean_text']` keeps the cleaned text of that
  meeting's row, and `.iloc[0]` takes it out as a string.
- `tokens.count(word)` counts the word in the list of tokens.
- `word in text.split()` is True when the word is one of a document's tokens; `sum()` of Trues and
  Falses counts the Trues, so `df_word` is the number of documents that use the word.
- `np.log` is the natural log.
- In the printout, `:2d` and `:3d` pad a whole number to 2 or 3 characters so the columns line up,
  and `:.6f` shows a number with 6 decimals.


In [ ]:
# GUIDED — run as-is
# Step 3, continued: the TF-IDF formula by hand, for two words in the 16 December 2008 minutes
doc_dec2008 = fomc_df.loc[fomc_df['date'] == '2008-12-16', 'clean_text'].iloc[0]
tokens = doc_dec2008.split()            # Part 2 joined the tokens with spaces; split() gets them back
n_docs = len(fomc_df)

for word in ['recession', 'inflation']:
    tf = tokens.count(word) / len(tokens)
    df_word = sum(word in text.split() for text in fomc_df['clean_text'])
    idf = np.log(n_docs / df_word)
    print(f'{word:10s} used {tokens.count(word):2d} times in {len(tokens)} tokens: TF {tf:.6f} | '
          f'in {df_word:3d} of {n_docs} documents: IDF {idf:.4f} | TF-IDF {tf * idf:.6f}')

**Expected output:**
```
recession  used  2 times in 4641 tokens: TF 0.000431 | in  87 of 243 documents: IDF 1.0272 | TF-IDF 0.000443
inflation  used 28 times in 4641 tokens: TF 0.006033 | in 243 of 243 documents: IDF 0.0000 | TF-IDF 0.000000
```
"inflation" is fourteen times as frequent in these minutes as "recession", yet it scores zero: every
set of minutes uses it, so it does nothing to tell one meeting from another. `max_df=0.85` removes such
terms from Step 3's matrix altogether: "inflation" is not one of its 5,000 terms.


## Part 4: Compute LM-Style Sentiment (this lab's lists) (YOUR TASK — 15 min)

The **Loughran-McDonald (LM) dictionary** is the standard for financial text sentiment. It was built
from companies' annual reports (10-K filings), after its authors showed that general-purpose word lists
misread financial text: they count words such as "liability" and "tax" as negative.

We compute, for each document:
- **Net Sentiment** = (positive count − negative count) / total tokens
- **Uncertainty** = uncertainty count / total tokens

where the total is the number of tokens Part 2 left in the document.

Step 4 does not use that dictionary. It uses three short lists inspired by Loughran-McDonald, a few
dozen words each, picked for this lab; they are not the LM master dictionary, which the link in the
cell leads to. `risk`, `uncertain` and `uncertainty` appear on both the negative and the uncertainty
lists. Two more things to notice:
- some words here, such as `growth` and `increase`, describe an economy rather than judge it;
- Part 2 lemmatized the tokens, so plural entries such as `losses` and `risks` never match: those
  tokens now read `loss` and `risk`.

`set([...])` turns a list into a set, so `t in lm_negative` is a fast lookup.


In [ ]:
# GUIDED — run as-is
# Step 4: three short word lists inspired by Loughran-McDonald
# The LM master dictionary: https://sraf.nd.edu/loughranmcdonald-master-dictionary/
lm_negative = set([
    'adverse', 'adversely', 'against', 'concern', 'concerned', 'concerns',
    'decline', 'declined', 'declining', 'decrease', 'decreased', 'deficit',
    'deteriorate', 'deteriorated', 'deteriorating', 'difficult', 'difficulty',
    'downturn', 'fail', 'failure', 'falling', 'loss', 'losses', 'negative',
    'negatively', 'recession', 'recessionary', 'risk', 'risks', 'risky',
    'severe', 'severely', 'slowdown', 'sluggish', 'stress', 'stressed',
    'threat', 'threaten', 'troubled', 'uncertain', 'uncertainty',
    'unfavorable', 'volatile', 'volatility', 'vulnerability', 'vulnerable',
    'weak', 'weaken', 'weakened', 'weakness', 'worse', 'worsen', 'worsened',
    'worsening', 'worst'
])

lm_positive = set([
    'achieve', 'achieved', 'achievement', 'benefit', 'beneficial', 'confidence',
    'confident', 'favorable', 'gain', 'gained', 'gains', 'good', 'growth',
    'improve', 'improved', 'improvement', 'improving', 'increase', 'increased',
    'opportunity', 'opportunities', 'optimism', 'optimistic', 'positive',
    'positively', 'profit', 'profitable', 'progress', 'prosper', 'prosperity',
    'rebound', 'recover', 'recovered', 'recovery', 'strength', 'strengthen',
    'strengthened', 'strong', 'stronger', 'strongest', 'success', 'successful',
    'upturn', 'upward'
])

lm_uncertainty = set([
    'approximate', 'approximately', 'assume', 'assumption', 'believe',
    'cautious', 'could', 'depend', 'depends', 'doubt', 'doubtful',
    'estimate', 'estimated', 'expect', 'expected', 'forecast', 'foreseeable',
    'indefinite', 'indefinitely', 'likelihood', 'may', 'might', 'nearly',
    'pending', 'perhaps', 'possible', 'possibly', 'predict', 'prediction',
    'preliminary', 'probable', 'probably', 'project', 'projected',
    'risk', 'roughly', 'seem', 'seemed', 'seemingly', 'suggest',
    'tentative', 'uncertain', 'uncertainty', 'unclear', 'unpredictable',
    'variable'
])

print(f'Words in the lists: {len(lm_negative)} negative, {len(lm_positive)} positive, '
      f'{len(lm_uncertainty)} uncertainty')

**Expected output:** `Words in the lists: 55 negative, 44 positive, 46 uncertainty`.

**The blanks.** `compute_lm_sentiment` scores one cleaned document. Fill in its eight blanks: the word
list each count looks in (the function's own arguments, not the notebook's `lm_` lists), and the two
formulas above, from the counts and the total the function has already computed.

`sum(1 for t in tokens if t in neg_words)` adds 1 for each token found in the list: a count. The
function gives back a dictionary, one entry per number.


In [ ]:
# YOUR TASK — fill in the eight blanks (____)
# Step 4, continued: count each list's words in one document, and turn the counts into shares
def compute_lm_sentiment(text, neg_words, pos_words, unc_words):
    """Net sentiment and uncertainty of one cleaned document, with the counts behind them."""
    tokens = text.split()
    total = len(tokens)

    neg_count = sum(1 for t in tokens if t in ____)
    pos_count = sum(1 for t in tokens if t in ____)
    unc_count = sum(1 for t in tokens if t in ____)

    net_sentiment = (____ - ____) / ____
    uncertainty = ____ / ____

    return {'net_sentiment': net_sentiment, 'uncertainty': uncertainty,
            'neg_count': neg_count, 'pos_count': pos_count,
            'unc_count': unc_count, 'total_words': total}

Step 4, continued scores every document and summarizes.
- `lambda x: compute_lm_sentiment(x, ...)` is a one-line function: `.apply` calls it on each document,
  with the three lists added.
- `pd.DataFrame(list_of_dictionaries)` makes one row per dictionary and one column per key.
- `fomc_df[sentiment_df.columns] = sentiment_df` adds those columns to `fomc_df`, or replaces them if
  you run the cell again.
- `.idxmin()` gives the row label of the smallest value, and `.loc[row, 'date']` that row's date.


In [ ]:
# GUIDED — run as-is
# Step 4, continued: score every document, and summarize
sentiment_results = fomc_df['clean_text'].apply(
    lambda x: compute_lm_sentiment(x, lm_negative, lm_positive, lm_uncertainty))
sentiment_df = pd.DataFrame(sentiment_results.tolist())
fomc_df[sentiment_df.columns] = sentiment_df

print('=== Sentiment Summary ===')
print(f'Mean net sentiment: {fomc_df["net_sentiment"].mean():.6f}')
print(f'Std net sentiment:  {fomc_df["net_sentiment"].std():.6f}')
print(f'Mean uncertainty:   {fomc_df["uncertainty"].mean():.6f}')
print()
print('Most negative meeting:', fomc_df.loc[fomc_df['net_sentiment'].idxmin(), 'date'].date())
print('Most positive meeting:', fomc_df.loc[fomc_df['net_sentiment'].idxmax(), 'date'].date())

**Expected output:** mean net sentiment `0.007020`, standard deviation `0.008662`, mean uncertainty `0.021718`,
most negative meeting `2008-12-16` and most positive `2004-06-30`. A mean of `-0.007020`, with the two
meetings swapped, means the counts in the net-sentiment formula are the wrong way round.

The mean is positive: in most meetings the minutes describe a growing economy, and words such as
`growth`, `increase` and `strong` are on the positive list.


## Part 5: Visualize Sentiment Over Time (GUIDED — 14 min)

Step 5 plots both scores for every meeting, with a 12-month rolling average, and marks four events
with dotted lines: the Lehman Brothers failure (15 September 2008), Bernanke's taper remarks (22 May
2013), the emergency COVID meeting (15 March 2020) and the first rate rise of 2022 (16 March). The
lines mark the dates of the events, drawn so you can check the series against them; they are not a
claim that the series moves at each one.

- `draw_series` draws one panel; it is a function because both panels are drawn the same way.
- `.rolling('365D').mean()` averages each meeting's score with those of the 365 days before it. It
  needs the dates as the row labels, hence `set_index('date')`.
- A dictionary such as `events` pairs each date with its label; `.items()` gives the pairs.


In [ ]:
# GUIDED — run as-is
# Step 5: each score over time, with a 12-month rolling average and four events marked
def draw_series(ax, column, color, title, ylabel):
    """One score per meeting, and its average over the previous 365 days."""
    ax.plot(fomc_df['date'], fomc_df[column], color=color, linewidth=0.8, marker='o', markersize=2)
    rolling = fomc_df.set_index('date')[column].rolling('365D').mean()
    ax.plot(rolling.index, rolling.values, color='firebrick', linewidth=2.0, label='12-month rolling avg')
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.legend()

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
draw_series(axes[0], 'net_sentiment', 'steelblue', "FOMC Minutes: LM-Style Net Sentiment (this lab's lists)",
            'Net Sentiment\n(positive − negative) / total')
axes[0].axhline(0, color='gray', linestyle='--', linewidth=0.5)
draw_series(axes[1], 'uncertainty', 'darkorange', "FOMC Minutes: LM-Style Uncertainty (this lab's lists)",
            'Uncertainty / Total Words')

events = {'2008-09-15': 'Lehman', '2013-05-22': 'Taper', '2020-03-15': 'COVID', '2022-03-16': 'Tightening'}
for date_str, label in events.items():
    date = pd.Timestamp(date_str)
    for ax in axes:
        ax.axvline(date, color='gray', linestyle=':', linewidth=0.8)
    axes[0].annotate(label, xy=(date, axes[0].get_ylim()[1] * 0.9), fontsize=8, ha='center')

plt.tight_layout()
plt.show()

Step 5, continued prints the numbers behind the chart. `groupby('year')[...].mean()` averages each
score within each year, and `.loc[[...]]` keeps the listed years. `value_counts()` counts how many
times each year occurs among the meetings below zero, and `.sort_index()` puts the years in order.


In [ ]:
# GUIDED — run as-is
# Step 5, continued: yearly averages around the events, and the extremes
yearly = fomc_df.groupby('year')[['net_sentiment', 'uncertainty']].mean()
print(yearly.loc[[2001, 2007, 2008, 2009, 2013, 2020, 2022, 2023, 2025]].round(4))
print(f"\nWhole sample: net sentiment {fomc_df['net_sentiment'].mean():.4f}, "
      f"uncertainty {fomc_df['uncertainty'].mean():.4f}")

below_zero = fomc_df.loc[fomc_df['net_sentiment'] < 0, 'year'].value_counts().sort_index()
print('Meetings with net sentiment below 0, by year:', ', '.join(f'{y}: {n}' for y, n in below_zero.items()))

lowest = fomc_df.loc[fomc_df['net_sentiment'].idxmin()]
most_uncertain = fomc_df.loc[fomc_df['uncertainty'].idxmax()]
print(f"Lowest net sentiment: {lowest['date'].date()} ({lowest['net_sentiment']:.4f})")
print(f"Highest uncertainty:  {most_uncertain['date'].date()} ({most_uncertain['uncertainty']:.4f})")

**Expected output:** nine yearly rows, from 2001 (net sentiment 0.0001, uncertainty 0.0228) to 2025
(-0.0021, 0.0251); a whole-sample mean of 0.0070 and 0.0217; the meetings below zero by year, starting
`2001: 7, 2008: 9, 2009: 7`; the lowest net sentiment on 2008-12-16 (-0.0192); and the highest
uncertainty on 2007-06-28 (0.0392).

Here is what this dictionary finds, read off those numbers:

- **2008–2009 financial crisis:** both yearly means are negative (−0.0037 and −0.0014), 16 of the 25
  meetings of those two years score below zero, and the single most negative meeting is **2008-12-16**
  (−0.0192, against a sample mean of +0.0070). It is not the only negative stretch: 7 of the 13
  meetings of 2001 and 6 of the 8 of 2025 score below zero, and 2025's yearly mean (−0.0021) is lower
  than 2009's.
- **2013 taper tantrum and 2020 COVID: no uncertainty spike.** 2013's mean uncertainty is **0.0220**
  against a sample mean of **0.0217**, a difference in the fourth decimal; the highest-uncertainty
  meeting in the whole sample is 2007-06-28. In 2020 net sentiment dips negative (−0.0017 for the
  year), but uncertainty **falls** to 0.0181, well *below* the sample mean: the minutes used fewer of
  the listed hedging words.
- **2022–2023 tightening:** net sentiment stays **positive** in 2022 (+0.0059) and is near zero in
  2023 (+0.0007). A dictionary that counts "growth", "strong" and "improve" reads a tight labour
  market as good news regardless of the policy stance.

The last two run against the story you would expect. That is the point of Part 5: **a
word-count dictionary measures words, not policy.** It has no model of what the Fed is doing, only of
which words appear.


---

## Write it yourself (required — 22 min)

This part is required. The check cell below it must print "Passed" before you submit.

**The task.** Step 3, continued computed the TF-IDF score of two words in one meeting's minutes, with
the formula from Part 3. Write a function `tfidf_score(word, doc, docs)` that gives back the TF-IDF
score of any word in any document:

- `word` is one word in lowercase, such as `'recession'`;
- `doc` is one cleaned document: a string of tokens separated by spaces, such as `doc_dec2008` or any
  entry of `fomc_df['clean_text']`;
- `docs` is the whole collection of cleaned documents, as a list or a Series of such strings; `doc`
  is one of them, and at least one of them uses `word`.

It gives back one number: the word's share of the tokens in `doc`, times the natural log of the number
of documents in `docs` divided by the number of them that use the word.

- It takes about five lines: the `def` line, the TF, the count of documents that use the word, the
  IDF, and the `return`.
- Work only from the three arguments: the check also calls your function on the minutes of another
  meeting, and on a made-up collection of five short documents.
- Give back the score itself: not a percent, not rounded.
- To test it yourself: `tfidf_score('recession', doc_dec2008, fomc_df['clean_text'])` should give
  Step 3, continued's 0.000443. Python may display it as `np.float64(0.000442...)`: that is just a
  number.

Then run the check cell. It compares your function with Step 3, continued, then tries it on the
minutes of 15 March 2020 and on the made-up documents. `assert` stops with its message when its
condition is False; the message tells you what to fix.


In [ ]:
# WRITE IT YOURSELF (required): tfidf_score(word, doc, docs)

In [ ]:
# CHECK — for "Write it yourself" above

# globals() holds every name defined so far in this notebook
if "tfidf_score" not in globals():
    print("Not written yet: write tfidf_score() in the cell above and run it, then run this check.")
elif not all(name in globals() for name in ["fomc_df", "doc_dec2008"]) or "clean_text" not in fomc_df:
    print("Run Parts 1-3 first, Step 3, continued included, then this check.")
else:
    # Scores recorded for this lab's saved minutes, cleaned as in Part 2, and for five made-up
    # documents; then what some common slips give for each.
    TOY = ['rate rate hike corporate bond rate',
           'labor market tight wage growth',
           'rate cut inflation outlook',
           'corporate credit spread widened',
           'inflation expectation anchored']
    RIGHT = {'recession': 0.00044264310479895055, 'coronavirus': 0.018521591020422556,
             'toy': 0.45814536593707755}
    SLIPS = {
        'recession': {'count': 2.0543066493719295, 'log10': 0.00019223745786670703,
                      'log2': 0.0006385990121771397, 'smooth': 0.0004394877874660614,
                      'smooth+1': 0.000870429394878257, 'plus1': 0.0008735847122111461,
                      'no log': 0.0012036644896685465, 'tf only': 0.00043094160741219563,
                      'idf only': 1.0271533246859648, 'occurrences': 0.00018278700328391165,
                      'substring df': 0.000413903016671964, 'chars': 5.449089255628461e-05,
                      'unique': 0.0016447611284002637, 'percent': 0.044264310479895055,
                      'recleaned': 0.00044321610558186185, 'tf by len(docs)': 0.00845393682869107,
                      'sklearn': 0.013535256121782505, 'lab matrix': 0.02302421448966791,
                      'log(N)/df': 2.7209065839162427e-05, 'inverted': -0.00044264310479895055},
        'coronavirus': {'count': 111.66667226212759, 'log10': 0.008043824776238336,
                        'log2': 0.026721007514537178, 'smooth': 0.017992130259963362,
                        'smooth+1': 0.023797404766515026, 'plus1': 0.02432686552697422,
                        'no log': 0.1410681705092055, 'tf only': 0.005805274506551667,
                        'idf only': 3.190476350346503, 'occurrences': 0.0032011387834787767,
                        'chars': 0.002301124575228792, 'unique': 0.07444444817475174,
                        'percent': 1.8521591020422556, 'recleaned': 0.018586330269994608,
                        'dec2008 tokens': 0.0, 'tf by len(docs)': 0.4595336307083441,
                        'sklearn': 0.34554805336123806, 'lab matrix': 0.4402663031912577,
                        'log(N)/df': 0.003188872955994679, 'inverted': -0.018521591020422556},
        'toy': {'count': 2.7488721956224653, 'log10': 0.1989700043360188, 'log2': 0.6609640474436812,
                'smooth': 0.34657359027997264, 'smooth+1': 0.8465735902799727,
                'plus1': 0.9581453659370776, 'no log': 1.25, 'tf only': 0.5,
                'idf only': 0.9162907318741551, 'occurrences': 0.11157177565710488,
                'substring df': 0.25541281188299536, 'substring tf': 0.6108604879161034,
                'chars': 0.08084918222419016, 'unique': 0.6872180489056163,
                'percent': 45.81453659370776, 'global docs': 0.0, 'global n_docs': 2.3999571313903014,
                'tf by len(docs)': 0.5497744391244931, 'sklearn': 0.8297368715155836,
                'log(N)/df': 0.40235947810852507, 'inverted': -0.4581453659370775},
    }
    MESSAGES = {
        'count': "Your TF is the word's count. Divide it by the number of tokens in doc: the TF is a share.",
        'log10': "Your IDF uses log base 10. The formula uses the natural log, as Step 3, continued does.",
        'log2': "Your IDF uses log base 2. The formula uses the natural log, as Step 3, continued does.",
        'smooth': ("Your IDF adds 1 to both numbers inside the log (smoothing, as scikit-learn does). "
                   "The formula of Part 3 has no smoothing."),
        'smooth+1': ("Your IDF is scikit-learn's: 1 added inside the log and 1 added to the log. The "
                     "formula of Part 3 adds nothing."),
        'plus1': "Your IDF adds 1 to the log. The formula of Part 3 does not.",
        'no log': "Your IDF is the ratio itself. Take its natural log.",
        'tf only': "That is the TF alone. Multiply it by the IDF.",
        'idf only': "That is the IDF alone. Multiply it by the TF.",
        'occurrences': ("Your document count adds up every use of the word in every document. Count each "
                        "document once if it uses the word, however many times."),
        'substring df': ("Your document count also finds the word inside longer words ('rate' inside "
                         "'corporate', 'recession' inside 'recessionary'): testing a word against a whole "
                         "string finds pieces of words. Test it against each document's tokens."),
        'substring tf': ("Your count also finds the word inside longer words ('rate' inside 'corporate'): "
                         "counting in the string doc counts pieces of words. Count it among doc's tokens."),
        'chars': "Your TF divides by the number of characters in doc. Divide by its number of tokens.",
        'unique': ("Your TF divides by the number of different tokens in doc. Divide by all its tokens, "
                   "repeats included."),
        'percent': "Your score is 100 times too large: give back the score itself, not a percent.",
        'recleaned': ("Your function cleans doc again. doc is already cleaned, and lemmatizing twice "
                      "changes a few tokens: use doc as it is given."),
        'dec2008 tokens': ("Your function gives 0 for 'coronavirus' in the March 2020 minutes, which use it "
                           "35 times. Does it count the word in its doc argument, or in Step 3's tokens?"),
        'same word': ("For 'coronavirus' in the March 2020 minutes your function gives the same score as "
                      "for 'recession' in them. Does it use its word argument, or a word typed in?"),
        'global docs': ("Given five made-up documents, your function answers as if it had all 243 sets of "
                        "minutes. Use the docs argument everywhere, not fomc_df."),
        'global n_docs': ("Given five made-up documents, your function divides by 243 documents. Count "
                          "the documents in its docs argument, not n_docs or a typed-in 243."),
        'tf by len(docs)': ("Your TF divides the count by the number of documents. Divide it by the number "
                            "of tokens in doc."),
        'sklearn': ("That is scikit-learn's TfidfVectorizer score, with its own TF, IDF and row scaling. "
                    "Compute the formula of Part 3 yourself, as Step 3, continued does."),
        'lab matrix': ("That is the word's entry in Step 3's tfidf_matrix: scikit-learn's variant, not the "
                       "formula of Part 3. Compute the formula yourself, as Step 3, continued does."),
        'log(N)/df': ("Your IDF takes the log of the number of documents and then divides: brackets. The "
                      "log is of the whole ratio, documents / documents that use the word."),
        'inverted': ("Your score is negative: the ratio inside the log is upside down. It is documents / "
                     "documents that use the word, which is at least 1."),
    }

    def call(word, doc, docs):
        """Run your tfidf_score; if it goes wrong, say where to look."""
        # try/except catches an error inside your function, so the message can name the cause
        try:
            out = tfidf_score(word, doc, docs)
        except Exception as e:
            msg = str(e)
            hint = "Read the error above: it says what went wrong."
            if "positional argument" in msg or "unexpected keyword" in msg:
                hint = "tfidf_score takes three arguments, in this order: word, doc, docs."
            elif isinstance(e, ZeroDivisionError) or "division by zero" in msg:
                hint = ("By your count, no document in docs uses the word, so the IDF divides by zero. "
                        "Check three things: the arguments come in the order word, doc, docs; the word is "
                        "compared as given, in lowercase like the documents; and your loop over docs runs "
                        "over whole documents, one at a time.")
            elif "'list' object has no attribute" in msg:
                hint = ("docs can be a plain Python list, which has no .str, .apply or .sum. Loop over it "
                        "with for, which works on a list and on a Series.")
            raise AssertionError(f"tfidf_score({word!r}, ...) stopped with an error:\n  {msg[:200]}\n"
                                 f"{hint}") from None
        assert out is not None, "tfidf_score gave back nothing. Does it end with a return line?"
        assert np.size(out) == 1, (f"tfidf_score gives back {np.size(out)} values. Give back one number: "
                                   "the TF-IDF score.")
        return float(np.ravel(out)[0])

    def near(a, b):
        return np.isclose(a, b, rtol=1e-6, atol=1e-15)

    def diagnose(got, case, want, what):
        """Fail with the message of the slip that gives got, or with a general message."""
        for slip, value in SLIPS[case].items():
            assert not near(got, value), MESSAGES[slip]
        assert got != 0, ("Your function gives 0: check the TF division (/ not //) and that you count among "
                          "doc.split().")
        # round(v, places) keeps that many decimals; try 1 to 8 of them
        for places in [1, 2, 3, 4, 5, 6, 7, 8]:
            assert not np.isclose(got, round(want, places), rtol=0, atol=1e-15), (
                "Your number is rounded. Give back the full number; round only when you print.")
        assert False, (f"{what}: your function gives {got:.6f}, but it should be {want:.6f}. Check three "
                       "things: the TF (a share of doc's tokens), the count of documents in docs that use "
                       "the word as a token, and the natural log of the ratio.")

    tokens_first = len(fomc_df['clean_text'].iloc[0].split())
    assert len(fomc_df) == 243 and tokens_first == 4832 and len(doc_dec2008.split()) == 4641, (
        "fomc_df or doc_dec2008 is not what Parts 1-3 build (243 documents; 4832 tokens in the first). "
        "Write only the function, rerun the notebook from the top, then this cell.")

    got = call('recession', doc_dec2008, fomc_df['clean_text'])
    if not near(got, RIGHT['recession']):
        diagnose(got, 'recession', RIGHT['recession'],
                 "For 'recession' in the 16 December 2008 minutes (Step 3, continued printed 0.000443)")

    doc_mar2020 = fomc_df.loc[fomc_df['date'] == '2020-03-15', 'clean_text'].iloc[0]
    got = call('coronavirus', doc_mar2020, fomc_df['clean_text'])
    if not near(got, RIGHT['coronavirus']):
        assert not near(got, call('recession', doc_mar2020, fomc_df['clean_text'])), MESSAGES['same word']
        diagnose(got, 'coronavirus', RIGHT['coronavirus'], "For 'coronavirus' in the 15 March 2020 minutes")

    got = call('rate', TOY[0], TOY)
    if not near(got, RIGHT['toy']):
        diagnose(got, 'toy', RIGHT['toy'], "For 'rate' in the first of the five made-up documents")

    print("Passed. tfidf_score reproduces Step 3, continued, and gives the right score on another "
          "meeting's minutes and on five made-up documents.")

    words = ['recession', 'coronavirus', 'tariff', 'inflation']
    meetings = ['2008-12-16', '2020-03-15', '2025-05-07']
    print(f"\nYour tfidf_score for four words in three meetings' minutes:\n  {'':12}"
          + ''.join(f'{m:>12}' for m in meetings))
    for w in words:
        row = [call(w, fomc_df.loc[fomc_df['date'] == m, 'clean_text'].iloc[0], fomc_df['clean_text'])
               for m in meetings]
        print(f'  {w:12}' + ''.join(f'{s:12.5f}' for s in row))

### Interpretation Question

1. In the table your check printed, which word scores highest in each of the three meetings? Could
   Part 4's word lists have flagged `coronavirus` or `tariff`? Say what TF-IDF finds in one meeting's
   minutes that a fixed word list cannot, and quote two numbers from the table.


*Your answer here:*

1.


---

## Part 6: Cluster Documents with K-Means on TF-IDF (EXTENSION, optional — 12 min)

This is the **Chapter 22 callback**: K-Means on the TF-IDF matrix, to see whether the minutes fall
into distinct "language regimes".

K-Means measures distances, and across 5,000 columns most distances between documents look alike, so
Step 6 first compresses the matrix to 50 columns with `TruncatedSVD`, the version of PCA that works on
a sparse matrix (it skips PCA's centring, which would fill in the zeros).
`explained_variance_ratio_.sum()` is the share of the variance the 50 columns keep.

**The blanks.** Fill in K-Means with **3 clusters**, run from **10 different random starts**; it keeps
the best of the 10 runs. `KMeans` takes both as settings, as in Lab 22.

Expect a lower silhouette than Lab 22's: distances between long documents differ less than distances
between countries' indicators do.

If you skip Part 6, leave its cells unrun, so your submitted notebook shows no error.


In [ ]:
# YOUR TASK (optional) — fill in the two blanks (____)
# Step 6: compress the TF-IDF matrix to 50 columns, cluster the documents, and score the clusters
svd = TruncatedSVD(n_components=50, random_state=42)
tfidf_reduced = svd.fit_transform(tfidf_matrix)
print(f'Explained variance (50 components): {svd.explained_variance_ratio_.sum():.1%}')

kmeans = KMeans(n_clusters=____, n_init=____, random_state=42)
fomc_df['cluster'] = kmeans.fit_predict(tfidf_reduced)

sil_score = silhouette_score(tfidf_reduced, fomc_df['cluster'])
print(f'Silhouette score (K={kmeans.n_clusters}): {sil_score:.3f}')

**Expected output:** `Explained variance (50 components): 74.4%` and `Silhouette score (K=3): 0.166`.

Step 6, continued draws the clusters on the first two principal components of the 50 columns, and
labels three meetings. `mask` is True for the documents of one cluster, and `coords_2d[mask, 0]` keeps
their first coordinate; `.to_numpy()` turns the True/False column into a plain array for that.


In [ ]:
# GUIDED — run as-is (optional)
# Step 6, continued: the clusters on two principal components, and what each holds
pca_2d = PCA(n_components=2, random_state=42)
coords_2d = pca_2d.fit_transform(tfidf_reduced)

fig, ax = plt.subplots(figsize=(10, 7))
for k in range(kmeans.n_clusters):
    mask = (fomc_df['cluster'] == k).to_numpy()
    ax.scatter(coords_2d[mask, 0], coords_2d[mask, 1], alpha=0.6, s=40, label=f'Cluster {k}')
for d in ['2008-12-16', '2020-03-15', '2022-06-15']:
    i = fomc_df.index[fomc_df['date'] == d][0]      # the row of that meeting
    ax.annotate(d[:7], xy=(coords_2d[i, 0], coords_2d[i, 1]), fontsize=8)
ax.set_xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]:.1%} variance)')
ax.set_ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]:.1%} variance)')
ax.set_title('FOMC Minutes — K-Means Clusters in PCA Space')
ax.legend()
plt.tight_layout()
plt.show()

print('=== Cluster Composition ===')
for k in range(kmeans.n_clusters):
    cluster_docs = fomc_df[fomc_df['cluster'] == k]
    print(f'Cluster {k}: {len(cluster_docs)} documents, '
          f'{cluster_docs["date"].min().date()} to {cluster_docs["date"].max().date()}')

**Expected output:** a chart with PC1 at 15.4% and PC2 at 9.2% of the variance, then
`Cluster 0: 27 documents, 2000-02-02 to 2022-01-26`, `Cluster 1: 88 documents, 2000-03-21 to
2009-12-15` and `Cluster 2: 128 documents, 2009-11-04 to 2026-07-29`.

**Your answer.** Clusters 1 and 2 split the sample around 2009, but Cluster 0's 27 documents are
spread from 2000 to 2022. Print their dates in a new cell, with
`fomc_df.loc[fomc_df['cluster'] == 0, 'date'].dt.date.tolist()`, and say what those meetings have in
common.


*Your answer here:*


---

## Challenge (Take-Home): Pre-2020 vs Post-2020 Sentiment Patterns

Did the Fed's language change after COVID-19? The cell below splits the meetings at March 2020 and
prints each period's means. Add a chart that compares the distribution of net sentiment in the two
periods (side-by-side histograms, box plots, or overlaid density plots). Then, in the text cell below
it, write 2-3 sentences: did the minutes become more negative? More uncertain? Quote your numbers, and
say which economic events could explain the shift.


In [ ]:
# YOUR TASK — take-home Challenge: a chart comparing the two periods
pre_covid = fomc_df[fomc_df['date'] < '2020-03-01']
post_covid = fomc_df[fomc_df['date'] >= '2020-03-01']

print('=== Pre-COVID (before March 2020) ===')
print(f'  Documents: {len(pre_covid)}')
print(f'  Mean net sentiment: {pre_covid["net_sentiment"].mean():.6f}')
print(f'  Mean uncertainty:   {pre_covid["uncertainty"].mean():.6f}')

print('\n=== Post-COVID (March 2020 onward) ===')
print(f'  Documents: {len(post_covid)}')
print(f'  Mean net sentiment: {post_covid["net_sentiment"].mean():.6f}')
print(f'  Mean uncertainty:   {post_covid["uncertainty"].mean():.6f}')

# YOUR CODE HERE: the chart

*Your answer here:*


---
## Digital Portfolio: Your README

Copy the prompt below into Claude or ChatGPT. Fill in each [YOUR VALUE] from your own output
first, and do not quote a number this notebook did not print. **Do NOT ask the AI to write
Python code — only documentation.**

```
I need help writing a project README for my data science lab.
**Important Rule:** Do NOT generate any Python code for me.

**What I did in this lab:**
* Loaded the minutes of [YOUR VALUE] FOMC meetings, [YOUR VALUE] to
  [YOUR VALUE], and cleaned the text: lowercase, letters only, stop
  words removed, words reduced to their dictionary form
* Built a TF-IDF matrix of [YOUR VALUE] documents by [YOUR VALUE]
  terms; the top terms by average TF-IDF were [YOUR VALUE]
* Scored each meeting's net sentiment and uncertainty with short word
  lists in the style of Loughran-McDonald: mean net sentiment
  [YOUR VALUE], most negative meeting [YOUR VALUE]
* Plotted both scores over time against four events: [ONE SENTENCE
  ON WHAT I FOUND]
* Wrote a function, tfidf_score(), that computes the TF-IDF score of a
  word in a document from the formula: [YOUR VALUE] for "coronavirus"
  in the 15 March 2020 minutes
* (Only if I did Part 6 or the Challenge) [YOUR RESULT]

**Please write a README.md entry including:**
1. Project Title: Reading the Fed: FOMC Minutes as Data
2. Objective: one sentence
3. Methodology: Bullet points of technical steps
4. Key Findings: Summary of results
Write it plainly, in the first person. No words like 'robust', 'comprehensive'
or 'leverage'. Use only the numbers I gave you.
```


---

## Submit this lab on GitHub

**Repository name for this lab:** `econ3916-lab23-fedspeak`

**First time?** Read the **GitHub Setup** page in the Start Here module once,
start to finish. It assumes you have never used git and never seen GitHub.

1. On [github.com](https://github.com): **+** (top right) → **New repository**.
   Name it exactly `econ3916-lab23-fedspeak`, tick **Add a README file**, and create it.
2. In Colab: **File → Save a copy in GitHub**. Choose `econ3916-lab23-fedspeak`, branch `main`,
   commit message `Lab 23 submission`. This pushes the notebook, and the
   notebook's rendered output — charts included — is what is graded.
3. Open the repo on github.com, click `README.md`, then the pencil icon, and
   paste in the README you drafted from the prompt above. **Commit
   changes.** Check every number in it against your own output first: this
   goes out under your name.
4. On Canvas: in Colab use **File → Download → Download .ipynb**, upload that
   file to the assignment, and paste your repository URL in the **Comments**
   box beside the upload. Then click **Submit**. Do not use the Website URL
   tab: Canvas keeps one submission, so a URL sent on its own replaces your
   notebook.

No terminal, no token, nothing to install.
